# 03 — Machine Learning Benchmark

Compare Logistic Regression, Random Forest, XGBoost and a compact PyTorch neural network under the same stratified split. Model selection is based on **validation PR-AUC**. The final test set is not scored in this notebook.

In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_curve, roc_curve
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import load_dataset, split_dataset
from src.evaluation import best_f1_threshold, classification_metrics
from src.models import FraudMLP, build_logistic_regression, build_random_forest, build_xgboost
from src.preprocessing import build_linear_preprocessor, build_tree_preprocessor

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

FIGURES = ROOT / 'outputs' / 'figures'
METRICS = ROOT / 'outputs' / 'metrics'
FIGURES.mkdir(parents=True, exist_ok=True)
METRICS.mkdir(parents=True, exist_ok=True)

In [ ]:
transactions = load_dataset(ROOT / 'data' / 'creditcard.csv')
X_train, X_validation, X_test, y_train, y_validation, y_test = split_dataset(transactions, random_state=SEED)
feature_names = X_train.columns.tolist()

linear_preprocessor = build_linear_preprocessor(feature_names)
tree_preprocessor = build_tree_preprocessor(feature_names)

X_train_linear = linear_preprocessor.fit_transform(X_train)
X_validation_linear = linear_preprocessor.transform(X_validation)
X_train_tree = tree_preprocessor.fit_transform(X_train)
X_validation_tree = tree_preprocessor.transform(X_validation)

class_counts = y_train.value_counts()
positive_weight = float(class_counts.loc[0] / class_counts.loc[1])
positive_weight

In [ ]:
def train_neural_network(X_train_array, y_train_series, X_validation_array, y_validation_series):
    torch.manual_seed(SEED)
    model = FraudMLP(input_dim=X_train_array.shape[1])
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_function = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([positive_weight], dtype=torch.float32))

    train_features = torch.tensor(X_train_array, dtype=torch.float32)
    train_labels = torch.tensor(y_train_series.to_numpy(), dtype=torch.float32)
    validation_features = torch.tensor(X_validation_array, dtype=torch.float32)

    loader = DataLoader(TensorDataset(train_features, train_labels), batch_size=4096, shuffle=True)
    best_state = deepcopy(model.state_dict())
    best_pr_auc = -np.inf
    epochs_without_improvement = 0

    for _ in range(12):
        model.train()
        for batch_features, batch_labels in loader:
            optimizer.zero_grad()
            loss = loss_function(model(batch_features), batch_labels)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            validation_scores = torch.sigmoid(model(validation_features)).numpy()
        validation_pr_auc = classification_metrics(y_validation_series, validation_scores)['pr_auc']

        if validation_pr_auc > best_pr_auc + 1e-5:
            best_pr_auc = validation_pr_auc
            best_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= 3:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        validation_scores = torch.sigmoid(model(validation_features)).numpy()
    return model, validation_scores

In [ ]:
models = {}
validation_scores = {}

logistic = build_logistic_regression(SEED)
logistic.fit(X_train_linear, y_train)
models['Logistic Regression'] = logistic
validation_scores['Logistic Regression'] = logistic.predict_proba(X_validation_linear)[:, 1]

random_forest = build_random_forest(SEED)
random_forest.fit(X_train_tree, y_train)
models['Random Forest'] = random_forest
validation_scores['Random Forest'] = random_forest.predict_proba(X_validation_tree)[:, 1]

xgboost = build_xgboost(SEED)
xgboost.set_params(scale_pos_weight=positive_weight)
xgboost.fit(X_train_tree, y_train)
models['XGBoost'] = xgboost
validation_scores['XGBoost'] = xgboost.predict_proba(X_validation_tree)[:, 1]

neural_network, neural_scores = train_neural_network(
    X_train_linear, y_train, X_validation_linear, y_validation
)
models['Neural Network'] = neural_network
validation_scores['Neural Network'] = neural_scores

In [ ]:
benchmark_rows = []
for model_name, scores in validation_scores.items():
    metrics = classification_metrics(y_validation, scores, threshold=0.5)
    benchmark_rows.append({'model': model_name, **metrics})

benchmark = pd.DataFrame(benchmark_rows).sort_values('pr_auc', ascending=False).reset_index(drop=True)
benchmark

In [ ]:
selected_model = benchmark.loc[0, 'model']
selected_threshold = best_f1_threshold(y_validation, validation_scores[selected_model])
classical_models = benchmark[benchmark['model'] != 'Neural Network'].reset_index(drop=True)
explainability_model = selected_model if selected_model != 'Neural Network' else classical_models.loc[0, 'model']

selection = {
    'primary_metric': 'validation_pr_auc',
    'selected_model': selected_model,
    'selected_model_validation_pr_auc': float(benchmark.loc[0, 'pr_auc']),
    'selected_threshold': float(selected_threshold),
    'threshold_selection_rule': 'maximum validation F1',
    'explainability_model': explainability_model,
    'test_evaluated': False,
}
selection

In [ ]:
figure, axis = plt.subplots(figsize=(8, 5))
for model_name, scores in validation_scores.items():
    precision, recall, _ = precision_recall_curve(y_validation, scores)
    pr_auc = benchmark.loc[benchmark['model'] == model_name, 'pr_auc'].iloc[0]
    axis.plot(recall, precision, label=f'{model_name} (AP={pr_auc:.3f})')
axis.set_xlabel('Recall')
axis.set_ylabel('Precision')
axis.set_title('Validation precision–recall curves')
axis.legend()
figure.tight_layout()
figure.savefig(FIGURES / '03_precision_recall_curves.png', dpi=180)
plt.show()

In [ ]:
figure, axis = plt.subplots(figsize=(8, 5))
for model_name, scores in validation_scores.items():
    false_positive_rate, true_positive_rate, _ = roc_curve(y_validation, scores)
    roc_auc = benchmark.loc[benchmark['model'] == model_name, 'roc_auc'].iloc[0]
    axis.plot(false_positive_rate, true_positive_rate, label=f'{model_name} (AUC={roc_auc:.3f})')
axis.plot([0, 1], [0, 1], linestyle='--', linewidth=1)
axis.set_xlabel('False positive rate')
axis.set_ylabel('True positive rate')
axis.set_title('Validation ROC curves')
axis.legend()
figure.tight_layout()
figure.savefig(FIGURES / '04_roc_curves.png', dpi=180)
plt.show()

In [ ]:
benchmark.to_csv(METRICS / 'model_benchmark_validation.csv', index=False)
with (METRICS / 'model_selection.json').open('w') as file:
    json.dump(selection, file, indent=2)

print(f"Selected model: {selected_model}")
print(f"Validation PR-AUC: {selection['selected_model_validation_pr_auc']:.4f}")
print(f"Frozen validation-selected threshold: {selected_threshold:.4f}")
print('Held-out test performance has not been evaluated.')

## Selection rule

The model with the highest validation PR-AUC is selected because the fraud class is extremely rare. The operating threshold is then frozen using validation F1. ROC-AUC, precision, recall and F1 remain supporting diagnostics. The held-out test set is intentionally untouched here and will be evaluated once after all model and threshold decisions are final.